### Importing the Libraries

We import the main libraries that we need for data analysis and visualization.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

### Loading the Dataset

We load the unemployment dataset and take a quick look at the first rows.


In [2]:
df = pd.read_csv("Unemployment in India.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (768, 7)


,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural


### Checking the Dataset

Here we check the data types, number of rows, and number of columns.


In [3]:
display(df.head())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset shape:")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

,Region,Date,Frequency,Estimated Unemployment Rate (%),Estimated Employed,Estimated Labour Participation Rate (%),Area
0,Andhra Pradesh,31-05-2019,Monthly,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,30-06-2019,Monthly,3.05,11755881.0,42.05,Rural
2,Andhra Pradesh,31-07-2019,Monthly,3.75,12086707.0,43.50,Rural
3,Andhra Pradesh,31-08-2019,Monthly,3.32,12285693.0,43.97,Rural
4,Andhra Pradesh,30-09-2019,Monthly,5.17,12256762.0,44.68,Rural



Data types:


,dtype
Region,str
Date,str
Frequency,str
Estimated Unemployment Rate (%),float64
Estimated Employed,float64
Estimated Labour Participation Rate (%),float64
Area,str



Dataset shape:
Rows    : 768
Columns : 7


### Basic Statistics

We use descriptive statistics to get a general idea about the numerical values in the dataset.


In [4]:
print("Raw descriptive summary:")
display(df.describe(include="all").T)

Raw descriptive summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,740,28,Andhra Pradesh,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,740,14,31-10-2019,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Frequency,740,2,Monthly,381,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Estimated Unemployment Rate (%),740.0,NaN,NaN,NaN,11.787946,10.721298,0.0,4.6575,8.35,15.8875,76.74
Estimated Employed,740.0,NaN,NaN,NaN,7204460.025676,8087988.429458,49420.0,1190404.5,4744178.5,11275489.5,45777509.0
Estimated Labour Participation Rate (%),740.0,NaN,NaN,NaN,42.630122,8.111094,13.33,38.0625,41.16,45.505,72.57
Area,740,2,Urban,381,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cleaning Text Values

We remove extra spaces from the text columns so the values are consistent.


In [5]:
for col in ["Region", "Date", "Frequency", "Area"]:
    if col in df.columns:
        df[col] = df[col].str.strip()

### Checking Missing Values and Duplicates

Before cleaning the data, we check if there are missing values or duplicate rows.


In [6]:
df.columns = [str(c).strip() for c in df.columns]

missing_before = df.isna().sum().sort_values(ascending=False)
duplicate_before = int(df.duplicated().sum())

print("Missing values before cleaning:")
display(missing_before.to_frame("Missing Values"))
print("Duplicate rows before cleaning:", duplicate_before)

Missing values before cleaning:


,Missing Values
Region,28
Date,28
Frequency,28
Estimated Unemployment Rate (%),28
Estimated Employed,28
Estimated Labour Participation Rate (%),28
Area,28


Duplicate rows before cleaning: 27


### Missing Values Before Cleaning

This heatmap gives us a quick visual look at the missing values in the dataset.


In [7]:
fig = px.imshow(
    df.isna().astype(int),
    aspect="auto",
    color_continuous_scale=["white", "#d95f59"],
    labels={"x": "Columns", "y": "Rows", "color": "Missing"},
    title="Missing-Value Map Before Cleaning")
fig.update_layout(height=450)
fig.show()

### Cleaning and Formatting the Data

We rename some columns, convert the date column, and make sure the unemployment rate is numeric.


In [8]:
df = df.rename(columns={
    "Region": "State",
    "Estimated Unemployment Rate (%)": "Unemployment_Rate",
    "Estimated Employed": "Employed",
    "Estimated Labour Participation Rate (%)": "Labour_Participation_Rate"})

df["State"] = df["State"].str.strip()
df["Frequency"] = df["Frequency"].str.strip()
df["Area"] = df["Area"].str.strip()

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df["Unemployment_Rate"] = pd.to_numeric(df["Unemployment_Rate"], errors="coerce")
df["Employed"] = pd.to_numeric(df["Employed"], errors="coerce")
df["Labour_Participation_Rate"] = pd.to_numeric(
    df["Labour_Participation_Rate"], errors="coerce")


before = len(df)
df = df.drop_duplicates()
df = df.dropna()

print("Rows removed:", before - len(df))
print("Rows remaining:", len(df))


Rows removed: 28
Rows remaining: 740


### Creating New Columns

We create year, month, and period columns. The period column helps us compare the data before and during COVID-19.


In [9]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%b")
df["Month_Label"] = df["Date"].dt.strftime("%b %Y")

df["Period"] = np.where(
    df["Date"] < pd.Timestamp("2020-03-01"),
    "Pre-COVID",
    "COVID period")

print("Final columns:")
print(df.columns.tolist())


Final columns:
['State', 'Date', 'Frequency', 'Unemployment_Rate', 'Employed', 'Labour_Participation_Rate', 'Area', 'Year', 'Month', 'Month_Name', 'Month_Label', 'Period']


### Checking the Cleaned Data

We check the cleaned dataset again and look at the date range, areas, and data frequency.


In [10]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDate range:")
print(df["Date"].min().date(), "to", df["Date"].max().date())

print("\nArea:")
print(df["Area"].value_counts())

print("\nFrequency:")
print(df["Frequency"].value_counts())


Missing values:
State                        0
Date                         0
Frequency                    0
Unemployment_Rate            0
Employed                     0
Labour_Participation_Rate    0
Area                         0
Year                         0
Month                        0
Month_Name                   0
Month_Label                  0
Period                       0
dtype: int64

Duplicate rows:
0

Date range:
2019-05-31 to 2020-06-30

Area:
Area
Urban    381
Rural    359
Name: count, dtype: int64

Frequency:
Frequency
Monthly    740
Name: count, dtype: int64


### Dataset Overview

This gives us a simple summary of the main numbers in the dataset.


In [11]:
overview = pd.DataFrame({"Metric": [
        "Rows",
        "States/Regions",
        "Months represented",
        "Average unemployment rate (%)",
        "Median unemployment rate (%)",
        "Average labour participation rate (%)",
        "Average employed"],

    "Value": [
        len(df),
        df["State"].nunique(),
        df["Date"].nunique(),
        df["Unemployment_Rate"].mean(),
        df["Unemployment_Rate"].median(),
        df["Labour_Participation_Rate"].mean(),
        df["Employed"].mean()] })

display(overview)


,Metric,Value
0,Rows,7.400000e+02
1,States/Regions,2.800000e+01
2,Months represented,1.400000e+01
3,Average unemployment rate (%),1.178795e+01
4,Median unemployment rate (%),8.350000e+00
5,Average labour participation rate (%),4.263012e+01
6,Average employed,7.204460e+06


### Monthly Unemployment Trend

This line chart shows how the average unemployment rate changed over time.


In [12]:
monthly_trend = (
    df.groupby("Date", as_index=False)["Unemployment_Rate"]
      .mean()
      .sort_values("Date"))

fig = px.line(
    monthly_trend,
    x="Date",
    y="Unemployment_Rate",
    markers=True,
    title="Average Monthly Unemployment Rate",
    labels={"Date": "Month", "Unemployment_Rate": "Average Unemployment Rate (%)"},
    template="plotly_white")

fig.add_vrect(
    x0=pd.Timestamp("2020-03-01"),
    x1=df["Date"].max(),
    fillcolor="red",
    opacity=0.08,
    line_width=0,
    annotation_text="COVID period",
    annotation_position="top left" )

fig.update_traces(hovertemplate="%{x|%b %Y}<br>Unemployment: %{y:.2f}%<extra></extra>")
fig.update_layout(
    height=520,
    hovermode="x unified",
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(buttons=[
            dict(count=6, label="6M", step="month", stepmode="backward"),
            dict(count=12, label="12M", step="month", stepmode="backward"),
            dict(step="all", label="All") ]) ) )
fig.show()


### Highest and Lowest Unemployment Rate

Here we find the month with the highest unemployment rate and the month with the lowest rate.


In [13]:
peak_row = monthly_trend.loc[monthly_trend["Unemployment_Rate"].idxmax()]
lowest_row = monthly_trend.loc[monthly_trend["Unemployment_Rate"].idxmin()]

display(Markdown(
    f"""
**Trend takeaway:** The highest monthly average unemployment rate in the cleaned dataset was
**{peak_row['Unemployment_Rate']:.2f}%** in **{peak_row['Date']:%b %Y}**, while the lowest was
**{lowest_row['Unemployment_Rate']:.2f}%** in **{lowest_row['Date']:%b %Y}**.
"""
))



**Trend takeaway:** The highest monthly average unemployment rate in the cleaned dataset was
**24.88%** in **May 2020**, while the lowest was
**8.87%** in **May 2019**.


### Average Unemployment by State

This chart compares the average unemployment rate across the different states.


In [14]:
state_avg = df.groupby("State")["Unemployment_Rate"].mean().sort_values(ascending=False)

fig = px.bar(
    x=state_avg.index,
    y=state_avg.values,
    title="Average Unemployment Rate by State",
    labels={
        "x": "State",
        "y": "Average Unemployment Rate (%)"
    }
)

fig.show()

### Top 15 States

To make the comparison easier to read, we focus on the 15 states with the highest average unemployment rates.


In [15]:
state_avg = df.groupby("State")["Unemployment_Rate"].mean()
state_avg = state_avg.sort_values().tail(15)

fig = px.bar(
    state_avg,
    x=state_avg.values,
    y=state_avg.index,
    orientation="h",
    title="Top 15 States by Average Unemployment Rate",
    labels={"x": "Average Unemployment Rate (%)", "y": "State"}
)

fig.show()

### Unemployment by Area

This box plot compares unemployment rates between the different areas.


In [16]:
area_summary = df.groupby("Area")["Unemployment_Rate"].agg(
    ["mean", "median", "count"]
).round(2)

print(area_summary)

fig = px.box(
    df,
    x="Area",
    y="Unemployment_Rate",
    color="Area",
    title="Unemployment Rate by Area"
)

fig.show()

        mean  median  count
Area                       
Rural  10.32    6.76    359
Urban  13.17    9.97    381


### Before vs. During COVID-19

Here we compare the unemployment rate before COVID-19 with the rate during the COVID period.


In [17]:
period_summary = (
    df.groupby("Period")["Unemployment_Rate"]
      .agg(["mean", "median", "min", "max", "count"])
      .reindex(["Pre-COVID", "COVID period"]))

display(period_summary.round(2))


,mean,median,min,max,count
Period,,,,,
Pre-COVID,9.51,7.12,0.0,34.69,536
COVID period,17.77,14.52,0.0,76.74,204


### COVID-19 Comparison

This box plot gives us a clearer visual comparison between the two periods.


In [18]:
fig = px.box(
    df,
    x="Period",
    y="Unemployment_Rate",
    color="Period",
    points="outliers",
    title="Unemployment Rate Before and During the COVID Period",
    labels={"Period": "Period", "Unemployment_Rate": "Unemployment Rate (%)"},
    template="plotly_white")
fig.update_layout(showlegend=False, height=520)
fig.show()


### Measuring the COVID-19 Change

We calculate the difference between the average unemployment rate before and during COVID-19.


In [19]:
pre_mean = period_summary.loc["Pre-COVID", "mean"]
covid_mean = period_summary.loc["COVID period", "mean"]

change = covid_mean - pre_mean
percent = (change / pre_mean) * 100

print(f"Before COVID: {pre_mean:.2f}%")
print(f"During COVID: {covid_mean:.2f}%")
print(f"Change: {change:.2f} percentage points")
print(f"Percentage change: {percent:.1f}%")

Before COVID: 9.51%
During COVID: 17.77%
Change: 8.26 percentage points
Percentage change: 86.9%


### State-Level COVID-19 Changes

Here we calculate the change for each state and find the states with the largest increases and decreases.


In [20]:
state_period = (
    df.groupby(["State", "Period"])["Unemployment_Rate"]
      .mean()
      .unstack())

state_period = state_period.dropna(subset=["Pre-COVID", "COVID period"])

state_period["Change_pp"] = state_period["COVID period"] - state_period["Pre-COVID"]

largest_increases = state_period.sort_values("Change_pp", ascending=False).head(10)
largest_decreases = state_period.sort_values("Change_pp", ascending=True).head(10)

display(
    pd.concat([
        largest_increases[["Pre-COVID", "COVID period", "Change_pp"]].rename_axis("State").reset_index().assign(Group="Largest increases"),
        largest_decreases[["Pre-COVID", "COVID period", "Change_pp"]].rename_axis("State").reset_index().assign(Group="Largest decreases")
    ]).round(2))


Period,State,Pre-COVID,COVID period,Change_pp,Group
0,Puducherry,1.59,38.96,37.36,Largest increases
1,Tamil Nadu,2.84,25.40,22.57,Largest increases
2,Jharkhand,14.28,36.35,22.07,Largest increases
3,Bihar,13.83,31.63,17.80,Largest increases
4,Karnataka,3.23,15.28,12.05,Largest increases
5,Haryana,22.94,34.65,11.72,Largest increases
6,Kerala,6.99,17.95,10.96,Largest increases
7,Telangana,4.66,15.44,10.79,Largest increases
8,Madhya Pradesh,4.74,14.07,9.33,Largest increases
9,Andhra Pradesh,5.04,13.58,8.54,Largest increases


### Largest Changes During COVID-19

This chart shows the states that had the largest changes in unemployment during the COVID period.


In [21]:
change_plot = (
    pd.concat([
        largest_increases.reset_index().assign(Group="Largest increases"),
        largest_decreases.reset_index().assign(Group="Largest decreases")
    ])
    .sort_values("Change_pp"))

fig = px.bar(
    change_plot,
    x="Change_pp",
    y="State",
    color="Group",
    orientation="h",
    text="Change_pp",
    title="States with the Largest COVID-Period Changes",
    labels={
        "Change_pp": "Change in unemployment (percentage points)",
        "State": "State / Region",
        "Group": "Ranking group"},
    template="plotly_white")
fig.update_traces(
    texttemplate="%{text:+.2f}",
    textposition="outside",
    hovertemplate="%{y}<br>Change: %{x:+.2f} percentage points<extra></extra>")
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(height=620)
fig.show()


### Labour Market Explorer

This chart looks at the relationship between labour participation and unemployment. The bubble size represents the number of employed people.


In [22]:
bubble_df = df.copy()


fig = px.scatter(
    bubble_df,
    x="Labour_Participation_Rate",
    y="Unemployment_Rate",
    size="Employed",
    color="Area",
    hover_name="State",
    animation_frame="Date",
    title="Labour Market Explorer",
    labels={
        "Labour_Participation_Rate": "Labour Participation Rate (%)",
        "Unemployment_Rate": "Unemployment Rate (%)",
        "Employed": "Employed"
    }
)

fig.show()

### Correlation Analysis

Here we calculate the correlation between unemployment, employment, and labour participation.


In [23]:
corr_cols = [
    "Unemployment_Rate",
    "Employed",
    "Labour_Participation_Rate"]

corr = df[corr_cols].corr()
display(corr.round(3))


,Unemployment_Rate,Employed,Labour_Participation_Rate
Unemployment_Rate,1.000,-0.223,0.003
Employed,-0.223,1.000,0.011
Labour_Participation_Rate,0.003,0.011,1.000


### Correlation Heatmap

This heatmap makes the correlations easier to compare visually.


In [24]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix of Labour-Market Indicators",
    labels={"color": "Correlation"})
fig.update_layout(height=500)
fig.show()


### Monthly Patterns and Seasonal Trends

Here we compare the average unemployment rate across the different months.

May 2020 had the highest average unemployment rate, while May 2019 had the lowest.

Because the dataset covers only 14 months, it is difficult to confirm a strong seasonal pattern.


In [25]:
monthly_pattern = (
    df.groupby("Month_Name")["Unemployment_Rate"]
      .mean()
      .sort_values(ascending=False)
      .reset_index()
)

monthly_pattern


,Month_Name,Unemployment_Rate
0,Apr,23.641569
1,May,16.646190
2,Mar,10.700577
3,Jun,10.553462
4,Feb,9.964717
5,Jan,9.950755
6,Oct,9.900909
7,Nov,9.868364
8,Aug,9.637925
9,Dec,9.497358


In [26]:
fig = px.bar(
    monthly_pattern,
    x="Month_Name",
    y="Unemployment_Rate",
    title="Average Unemployment Rate by Month",
    labels={
        "Month_Name": "Month",
        "Unemployment_Rate": "Average Unemployment Rate (%)"
    }
)

fig.show()


### Policy and Social Insights

The analysis shows that unemployment increased during the COVID-19 period.

The average unemployment rate increased from 9.51% before COVID-19 to 17.77% during the COVID-19 period.

This suggests that economic shocks can have a strong effect on employment.

### Key Insights

- Governments can provide employment support during economic crises.
- Areas with higher unemployment may need more targeted support.
- Monitoring unemployment rates can help identify problems early.
- Training and job opportunities can help people return to work after economic shocks.
